####1. Set catalog and schema

> Make sure you are using a catalog and schema from your environment.\
The same catalog and schema shown in this notebook may not exist in your account and will throw an error.

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

####2. Create the events dataframe 

In [0]:
from pyspark.sql import Row
from datetime import datetime, timedelta
import random

# Build a small batch of event rows inline — no external data needed
base_time = datetime(2024, 1, 15, 9, 0, 0)
batch_1 = [
    Row(event_id=i,
        event_type=random.choice(["click", "view", "purchase"]),
        user_id=random.randint(100, 199),
        event_ts=base_time + timedelta(seconds=i * 30))
    for i in range(1, 11)
]

df_batch_1 = spark.createDataFrame(batch_1)
display(df_batch_1)

####3. Write the first batch

In [0]:
df_batch_1.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("events")

print("Batch 1 written — 10 rows appended.")

> We just wrote the first batch of ten rows into a Delta table called events. \
The mode is append — new rows are added alongside whatever is already in the table. In this case the table was empty, so these are the first ten rows.

####4. Create a second batch of events

In [0]:
base_time_2 = datetime(2024, 1, 15, 14, 0, 0)
batch_2 = [
    Row(event_id=i,
        event_type=random.choice(["click", "view", "purchase"]),
        user_id=random.randint(200, 299),
        event_ts=base_time_2 + timedelta(seconds=i * 30))
    for i in range(11, 21)
]

df_batch_2 = spark.createDataFrame(batch_2)
df_batch_2.display()

####5. Write a second batch with overwrite

In [0]:
df_batch_2.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("events")

print("Batch 2 written — table overwritten with 10 new rows.")

> This time the mode is overwrite. The table now contains only these ten rows — the first batch is gone.\
The important thing to understand about overwrite in Delta is that it is atomic. It is not a file delete followed by a write. Under the hood, Delta marks the old Parquet files as removed and adds the new ones in a single transaction log commit. A reader querying the table at the exact moment of the overwrite will always see either the old complete state or the new complete state — never a mix.

####6. Confirm with DESCRIBE HISTORY

In [0]:
%sql
DESCRIBE HISTORY events;

> Two commits — version 0 for the first write, version 1 for the overwrite. Each is a single atomic transaction. This is the transaction log making overwrites safe.

####7. Batch read: path-based

> The below approach is a valid approach but it will not work.\
Reason: The S3 location is managed by Unity Catalog and no one can access it directly.

In [0]:
location = spark.sql("DESCRIBE DETAIL events") \
                .select("location") \
                .collect()[0][0]

print(f"Delta table path: {location}")                

In [0]:
df_path = spark.read.format("delta").load(location)
print(f"Reading from: {location}")
print(f"Row count: {df_path.count()}")
display(df_path)

> spark reads directly at the S3 location of the External Delta tables only.\
Permission to access the location must be given by Unity Catalog.\
This is useful when you are reading from external tables in a shared bucket, or any situation where the catalog is not available.

####8. Batch read: catalog-based

In [0]:
df_catalog = spark.table("events")
print(f"Row count: {df_catalog.count()}")
display(df_catalog)

> Cleaner, shorter, and governed — access control and lineage tracking apply automatically. \
The result is identical. In Databricks pipelines, this is the one you will use by default. \
Use path-based only when you have a specific reason to bypass the catalog.